# MFT and FFT propagation: Fraunhofer and Fresnel

This notebook exercises the production 2 × 2 propagation API. **MFT versus FFT selects the numerical transform; Fraunhofer versus Fresnel selects the physical regime.** Fraunhofer is the default regime for both methods.

In [ ]:
import matplotlib.pyplot as plt
import torch

from fiatlux import (
    FFTPropagator, Field, Grid, MFTPropagator, PropagationRegime, Spectrum
)
from fiatlux.core.spectrum import Band

torch.set_default_dtype(torch.float64)

wavelength = 632.8e-9
propagation_scale = 0.20  # focal length for Fraunhofer, distance for Fresnel
n = 65
dx = 20e-6
input_grid = Grid(nx=n, ny=n, dx=dx, dy=dx, dtype=torch.float64)
output_dx = wavelength * propagation_scale / (n * dx)
output_grid = Grid(
    nx=n, ny=n, dx=output_dx, dy=output_dx, dtype=torch.float64
)

x, y = input_grid.meshgrid()
r2 = x.square() + y.square()
waist = 0.18e-3
amplitude = torch.exp(-r2 / waist**2).to(torch.complex128)
amplitude /= (amplitude.abs().square().sum() * input_grid.dx * input_grid.dy).sqrt()
spectrum = Spectrum(
    magnitude=0, band=Band(wavelength, 0.0, 1.0), samples=1, dtype=torch.float64
)
input_field = Field(amplitude.unsqueeze(0), input_grid, spectrum)

print(f"input sampling:  {input_grid.dx * 1e6:.2f} µm")
print(f"natural output sampling: {output_grid.dx * 1e6:.2f} µm")

## Numerical transforms

The MFT evaluates the Fourier integral on an explicit `output_grid`. The FFT constructs its natural conjugate grid, whose sampling obeys \(dx_2=\lambda q/(n_x dx_1)\). On that common grid the complex results must agree.

In [ ]:
fraunhofer_mft_propagator = MFTPropagator(
    focal_length=propagation_scale, output_grid=output_grid
)
fraunhofer_fft_propagator = FFTPropagator(focal_length=propagation_scale)

fresnel_mft_propagator = MFTPropagator(
    output_grid=output_grid,
    propagation=PropagationRegime.FRESNEL,
    distance=propagation_scale,
)
fresnel_fft_propagator = FFTPropagator(
    propagation=PropagationRegime.FRESNEL, distance=propagation_scale
)

## The four combinations

Fraunhofer transforms the input field directly. Fresnel applies the same MFT or FFT to the input field multiplied by a quadratic phase, then applies the output quadratic phase and carrier.

In [ ]:
fraunhofer_mft = fraunhofer_mft_propagator.apply(input_field)
fraunhofer_fft = fraunhofer_fft_propagator.apply(input_field)
fresnel_mft = fresnel_mft_propagator.apply(input_field)
fresnel_fft = fresnel_fft_propagator.apply(input_field)

## Quantitative checks

For matching natural sampling, MFT and FFT must agree as complex fields in both regimes. All four propagated fields must preserve integrated flux.

In [ ]:
torch.testing.assert_close(
    fraunhofer_mft.complex_amplitude, fraunhofer_fft.complex_amplitude,
    rtol=1e-10, atol=1e-10,
)
torch.testing.assert_close(
    fresnel_mft.complex_amplitude, fresnel_fft.complex_amplitude,
    rtol=1e-10, atol=1e-10,
)

def flux(field):
    return field.intensity().sum() * field.grid.dx * field.grid.dy

input_flux = flux(input_field)
for name, field in {
    "MFT Fraunhofer": fraunhofer_mft,
    "FFT Fraunhofer": fraunhofer_fft,
    "MFT Fresnel": fresnel_mft,
    "FFT Fresnel": fresnel_fft,
}.items():
    output_flux = flux(field)
    torch.testing.assert_close(output_flux, input_flux, rtol=1e-10, atol=1e-12)
    print(f"{name:17s}: flux = {float(output_flux):.12f}")

## Visual validation

In [ ]:
fields = [fraunhofer_mft, fraunhofer_fft, fresnel_mft, fresnel_fft]
titles = [
    "MFT · Fraunhofer", "FFT · Fraunhofer",
    "MFT · Fresnel", "FFT · Fresnel",
]
fig, axes = plt.subplots(2, 2, figsize=(9, 8), constrained_layout=True)
for ax, field, title in zip(axes.flat, fields, titles):
    extent_mm = [
        float(field.grid.x.min() * 1e3), float(field.grid.x.max() * 1e3),
        float(field.grid.y.min() * 1e3), float(field.grid.y.max() * 1e3),
    ]
    image = ax.imshow(
        field.intensity()[0].detach().cpu().numpy(),
        origin="lower", extent=extent_mm,
    )
    ax.set_title(title)
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    fig.colorbar(image, ax=ax)
plt.show()

Fraunhofer remains the default physical regime for both production propagators. MFT accepts a common explicit output grid and supports polychromatic fields. FFT currently accepts monochromatic fields because its natural physical output grid depends on wavelength.